# 02 - Chunking

**Project:** AI-Powered Chatbot for Data Engineering Students (FYP, UiTM)
**Phase 6, Step 2:** Knowledge Base Preparation - split the cleaned text into chunks and attach metadata.

## What this notebook does
1. Reads the cleaned text from `data/cleaned/` (output of `01_extraction.ipynb`) and the `extraction_manifest.json`.
2. Splits every source into **chunks** following the strategy below.
3. Attaches structural metadata to every chunk: source, licence, source type, section, page or file, difficulty.
4. Prints statistics and **real example chunks** from several sources so the chunk shape can be reviewed.
5. Writes the chunks to `data/chunks/chunks.json`.

All sizes and rules come from `config/settings.py`; nothing is hardcoded here.

## Chunking strategy
| Material | Unit that is split first | Then |
|---|---|---|
| **PDF textbooks** (S01, S02, S04, S07) | Headings found with regexes from `settings.PDF_HEADING_PATTERNS` (chapter or numbered section) | Sections longer than the chunk size are split into fixed-size chunks |
| **Lecture slides** (S03) | One slide = one section (the slide title is its heading) | Very short slides are merged into the next slide |
| **Markdown prose** (S05, S06 `.md`) | Markdown headings `#` to `###` (headings inside code fences are ignored) | Sections longer than the chunk size are split into fixed-size chunks |
| **Code** (S06 `.py`, `.sql`) | Python: top-level `def` / `class`; SQL: statements ending in `;` | Units are packed together up to the code chunk size. **No paragraph logic is applied to code** |

**Fixed-size fallback:** only when a section is still too long. Chunks are at most `CHUNK_SIZE` characters with `CHUNK_OVERLAP` characters of overlap, and the splitter prefers to cut at a paragraph, then a sentence, then a word boundary. Inside Markdown it also avoids cutting through a code fence where possible.

**Small sections:** a section shorter than `MIN_CHUNK_CHARACTERS` is merged into the next section of the same file.

**Same-section merge:** at the end, neighbouring prose chunks that belong to the **same section** (same file, same heading) are merged as long as the result stays within `CHUNK_SIZE`. Chunks of different sections are never merged just to reach a size target, so every chunk still sits under one heading.

## Labelling (section 9)
After the chunk shape was approved, every chunk is labelled with:
- **topic**: one of the eight course topics, chosen by embedding similarity between the chunk and a short description of each topic (`settings.TOPIC_DESCRIPTIONS`); chunks that match no topic well are labelled `Other`;
- **keywords**: the terms with the highest TF-IDF score in the chunk.

## 1. Setup

In [1]:
import sys
import re
import json
import statistics
from bisect import bisect_right
from pathlib import Path

import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter

# The notebook lives in notebooks/, so walk up until the folder that holds config/ is found.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "config").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from config import settings  # all sizes, paths and rules come from here

settings.CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

manifest = json.loads((settings.CLEANED_DIR / "extraction_manifest.json").read_text(encoding="utf-8"))

print("Cleaned text :", settings.CLEANED_DIR)
print("Chunks output:", settings.CHUNKS_FILE)
print(f"Prose chunks : up to {settings.CHUNK_SIZE} characters, overlap {settings.CHUNK_OVERLAP}, "
      f"sections under {settings.MIN_CHUNK_CHARACTERS} characters are merged")
print(f"Code chunks  : up to {settings.CODE_CHUNK_SIZE} characters")
print("Manifest entries:", len(manifest))

Cleaned text : C:\Users\thaf\OneDrive\Degree\FYP\fyp-data-engineering-chatbot\data\cleaned
Chunks output: C:\Users\thaf\OneDrive\Degree\FYP\fyp-data-engineering-chatbot\data\chunks\chunks.json
Prose chunks : up to 800 characters, overlap 100, sections under 150 characters are merged
Code chunks  : up to 1500 characters
Manifest entries: 218


## 2. Shared helpers
Everything works on **sections**: a section is a dict with a `label` (the heading), the `text`, and `start` (the character offset in the document, used to find the page of a PDF chunk). Sections always cut a document into contiguous pieces, so merging neighbours simply joins their text.

In [2]:
def make_splitter(chunk_size):
    """Fixed-size splitter (used only inside sections that are still too long).

    It cuts at a paragraph if it can, otherwise at a sentence, a line or a word, and shares
    CHUNK_OVERLAP characters between neighbouring chunks. add_start_index lets us find the page of a chunk.
    """
    return RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=settings.CHUNK_OVERLAP,
        separators=settings.TEXT_SPLIT_SEPARATORS,
        keep_separator="end",
        add_start_index=True,
    )


def merge_small_sections(sections, min_chars):
    """Merge every section shorter than min_chars into the NEXT section (or the previous one if it is last).

    The merged section keeps the label of the small heading it starts with (and names the second heading
    too). Text that comes before the first heading of a document is not a heading, so it takes the label
    of the section it is merged into.
    """
    merged, pending = [], None
    for sec in sections:
        if pending is not None:
            if pending.get("preamble"):
                label = sec["label"]
            elif sec["label"] != pending["label"]:
                label = f"{pending['label']} / {sec['label'].split(' > ')[-1]}"
            else:
                label = pending["label"]
            sec = {"label": label, "start": pending["start"], "text": pending["text"] + sec["text"]}
            pending = None
        if len(sec["text"].strip()) < min_chars:
            pending = sec
        else:
            merged.append(sec)
    if pending is not None:
        if merged:
            last = merged.pop()
            merged.append({"label": last["label"], "start": last["start"], "text": last["text"] + pending["text"]})
        else:
            merged.append(pending)
    return merged


def separate_heading(text):
    """The first line of a section is its heading. Returns (heading, offset where the body starts)."""
    first_line = text.split("\n", 1)[0]
    if 0 < len(first_line.strip()) <= 120 and len(text) > len(first_line) + 1:
        return first_line.strip(), len(first_line) + 1
    return "", 0


def tidy_pieces(pieces):
    """Clean up the pieces produced by splitting ONE long section.

    - drop pieces with no letter or digit (a lone '---' or code fence);
    - drop a piece that is entirely contained in the previous one (only an overlap copy);
    - merge a piece shorter than MIN_CHUNK_CHARACTERS into the previous one when it fits,
      without repeating the overlapping text.
    """
    tidy = []
    for text, start in pieces:
        if not re.search(r"\w", text):
            continue
        if tidy and text in tidy[-1][0]:
            continue
        if tidy and len(text) < settings.MIN_CHUNK_CHARACTERS:
            previous = tidy[-1][0]
            overlap = next((k for k in range(min(len(text), len(previous), settings.CHUNK_OVERLAP), 0, -1)
                            if previous.endswith(text[:k])), 0)
            joined = previous + (text[overlap:] if overlap else "\n" + text)
            if len(joined) <= settings.CHUNK_SIZE:
                tidy[-1] = (joined, tidy[-1][1])
                continue
        tidy.append((text, start))
    return tidy


def split_long_section(section):
    """Return [(chunk_text, start_offset)] for one section of prose (PDF text, slides, Markdown without code).

    A section that fits in CHUNK_SIZE stays a single chunk. A longer one is cut into fixed-size chunks with
    overlap. The heading line is taken off before splitting and put back on the first chunk, so a heading
    is never left alone in a chunk of its own.
    """
    text = section["text"]
    if len(text.strip()) <= settings.CHUNK_SIZE:
        return [(text.strip(), section["start"])]
    heading, body_start = separate_heading(text)
    reserve = len(heading) + 2 if heading else 0
    pieces = []
    for doc in make_splitter(settings.CHUNK_SIZE - reserve).create_documents([text[body_start:]]):
        piece = doc.page_content.strip()
        if piece:
            pieces.append((piece, section["start"] + body_start + doc.metadata["start_index"]))
    if heading and pieces:
        pieces[0] = (heading + "\n\n" + pieces[0][0], section["start"])
    return tidy_pieces(pieces)


def split_by_lines(unit, max_chars):
    """Cut an oversized block of text at a blank line if possible, otherwise at a line boundary."""
    pieces, buffer = [], []
    size = 0
    for line in unit.splitlines(keepends=True):
        if size + len(line) > max_chars and buffer:
            blank = max((i for i, l in enumerate(buffer) if not l.strip()), default=None)
            cut = blank + 1 if blank is not None and blank > 0 else len(buffer)
            pieces.append("".join(buffer[:cut]))
            buffer = buffer[cut:]
            size = sum(len(l) for l in buffer)
        buffer.append(line)
        size += len(line)
    if buffer:
        pieces.append("".join(buffer))
    return pieces


def new_chunk(source_id, label, text, page=None, file=None, source_type=None):
    """Build one chunk record with the metadata fields listed in settings.CHUNK_METADATA_FIELDS."""
    source = settings.SOURCES[source_id]
    return {
        "chunk_id": None,                                   # numbered once all chunks of the source are known
        "source_id": source_id,
        "source_name": source["name"],
        "licence": source["licence"],
        "source_type": source_type or source["source_type"],
        "section": label,
        "page": page,
        "file": file,
        "topic": None,                                      # labelled later, after the chunk shape is approved
        "keywords": None,
        "difficulty": settings.DIFFICULTY_BY_SOURCE[source_id],
        "text": text,
        "n_chars": len(text),
    }


dropped_chunks = []      # (source, file, text) of chunks removed because they hold no content
merge_counts = {}        # source -> number of same-section merges


def merge_same_section(source_id, chunks):
    """Merge neighbouring prose chunks that belong to the SAME section (same file and same heading)
    while the merged text stays within CHUNK_SIZE.

    Chunks of different sections are never merged, and code chunks are left as they are.
    If the second chunk starts with an overlap copy of the end of the first, that copy is not repeated.
    """
    merged = []
    for chunk in chunks:
        previous = merged[-1] if merged else None
        if (previous is not None and chunk["source_type"] != "code"
                and previous["source_type"] == chunk["source_type"]
                and previous["file"] == chunk["file"] and previous["section"] == chunk["section"]):
            a, b = previous["text"], chunk["text"]
            overlap = next((k for k in range(min(len(a), len(b), settings.CHUNK_OVERLAP), 0, -1)
                            if a.endswith(b[:k])), 0)
            joined = a + (b[overlap:] if overlap else "\n\n" + b)
            if len(joined) <= settings.CHUNK_SIZE:
                previous["text"], previous["n_chars"] = joined, len(joined)
                merge_counts[source_id] = merge_counts.get(source_id, 0) + 1
                continue
        merged.append(chunk)
    return merged


def number_chunks(source_id, chunks):
    """Finish the chunks of one source and give them ids like S02-0001, S02-0002, ... in reading order.

    First, neighbouring chunks of the same section are merged (merge_same_section).
    A trailing horizontal rule ('---') is removed. A chunk shorter than DROP_CHUNKS_BELOW characters is
    appended to the previous chunk of the same file when it fits, otherwise it is dropped (for example an
    index page that only says '# Streaming ## Units').
    """
    kept = []
    for chunk in merge_same_section(source_id, chunks):
        chunk["text"] = re.sub(r"\n+-{3,}\s*$", "", chunk["text"]).strip()
        chunk["n_chars"] = len(chunk["text"])
        if chunk["n_chars"] >= settings.DROP_CHUNKS_BELOW:
            kept.append(chunk)
        elif (kept and kept[-1]["file"] == chunk["file"] and kept[-1]["source_type"] == chunk["source_type"]
              and kept[-1]["n_chars"] + 2 + chunk["n_chars"] <= settings.CHUNK_SIZE):
            kept[-1]["text"] += "\n\n" + chunk["text"]          # real content: keep it with the previous chunk
            kept[-1]["n_chars"] = len(kept[-1]["text"])
        else:
            dropped_chunks.append((source_id, chunk["file"], chunk["text"]))
    for number, chunk in enumerate(kept, start=1):
        chunk["chunk_id"] = f"{source_id}-{number:04d}"
    return kept

## 3. PDF sources: textbooks (S01, S02, S04, S07) and slides (S03)
The cleaned PDF text has one marker line `=== page N ===` per page. A document is rebuilt from the pages, remembering where each page starts, so that any chunk can be mapped back to its page.

- **Textbooks:** headings are lines that match `settings.PDF_HEADING_PATTERNS` (for example `1.2 Data Science in Practice`).
- **Slides:** each slide is a section and its first line is its title.

In [3]:
def read_pdf_pages(source_id):
    """Read data/cleaned/<id>.txt and return [(page_number, page_text)] using the '=== page N ===' markers."""
    text = (settings.CLEANED_DIR / f"{source_id}.txt").read_text(encoding="utf-8")
    parts = re.split(r"=== page (\d+) ===\n", text)
    return [(int(number), body.strip("\n")) for number, body in zip(parts[1::2], parts[2::2])]


def build_document(pages):
    """Join the pages into one text and remember the offset where each page starts."""
    text, page_starts = "", []
    for number, body in pages:
        page_starts.append((len(text), number))
        text += body + "\n"
    return text, page_starts


def page_at(offset, page_starts):
    """Page number that contains the character at the given offset."""
    index = bisect_right([start for start, _ in page_starts], offset) - 1
    return page_starts[max(index, 0)][1]


def split_textbook_sections(text, patterns, first_label):
    """Cut a textbook into sections at every line that matches one of the heading patterns."""
    regexes = [re.compile(p) for p in patterns]
    cuts = []                                             # (offset, heading text)
    for match in re.finditer(r"^.*$", text, re.MULTILINE):
        line = match.group(0).strip()
        if line and any(r.match(line) for r in regexes):
            cuts.append((match.start(), line))
    sections, previous_start, label, is_preamble = [], 0, first_label, True
    for offset, heading in cuts:
        if offset > previous_start:
            sections.append({"label": label, "start": previous_start, "text": text[previous_start:offset],
                             "preamble": is_preamble})
        previous_start, label, is_preamble = offset, heading, False
    sections.append({"label": label, "start": previous_start, "text": text[previous_start:], "preamble": is_preamble})
    return [s for s in sections if s["text"].strip()]


def chunk_textbook(source_id):
    """Chunk one PDF textbook: split at headings first, fixed-size fallback inside long sections."""
    pages = read_pdf_pages(source_id)
    text, page_starts = build_document(pages)
    sections = split_textbook_sections(text, settings.PDF_HEADING_PATTERNS[source_id],
                                       first_label=f"{source_id} (before the first heading)")
    sections = merge_small_sections(sections, settings.MIN_CHUNK_CHARACTERS)
    chunks = []
    for sec in sections:
        for piece, start in split_long_section(sec):
            chunks.append(new_chunk(source_id, sec["label"], piece, page=page_at(start, page_starts),
                                    file=f"{source_id}.txt"))
    return number_chunks(source_id, chunks)


def chunk_slides(source_id):
    """Chunk lecture slides: one slide is one section, its first line is the title."""
    pages = read_pdf_pages(source_id)
    text, page_starts = build_document(pages)
    sections, offset = [], 0
    for number, body in pages:
        first_line = next((l.strip() for l in body.splitlines() if l.strip()), f"slide {number}")
        sections.append({"label": first_line, "start": offset, "text": body + "\n"})
        offset += len(body) + 1
    sections = merge_small_sections(sections, settings.MIN_CHUNK_CHARACTERS)
    chunks = []
    for sec in sections:
        for piece, start in split_long_section(sec):
            chunks.append(new_chunk(source_id, sec["label"], piece, page=page_at(start, page_starts),
                                    file=f"{source_id}.txt"))
    return number_chunks(source_id, chunks)

In [4]:
all_chunks = {}
for sid in ["S01", "S02", "S04", "S07"]:
    all_chunks[sid] = chunk_textbook(sid)
all_chunks["S03"] = chunk_slides("S03")

for sid in ["S01", "S02", "S03", "S04", "S07"]:
    print(f"{sid}: {len(all_chunks[sid]):4d} chunks")

S01:  319 chunks
S02:  392 chunks
S03:  119 chunks
S04:  354 chunks
S07:   23 chunks


## 4. Markdown sources: S05 wiki pages and S06 course notes
Each Markdown file is cut at its headings (`#` to `###`). Lines inside code fences are never treated as headings (a `# comment` in a bash block is not a heading). The `section` label is the path of headings, for example `03-data-warehouse > BigQuery > Partitioning`.

When a section is longer than the chunk size, **fenced code blocks are never torn apart**: prose is cut into fixed-size chunks with overlap, a code block stays whole (or, if it is longer than the chunk size, is split into smaller blocks that each carry their own fences), and a short paragraph is packed together with the code next to it so that an explanation stays with its code. A heading is never left alone in a chunk: it is kept with the text that follows it.

- S05: the first part of the label is the wiki page name.
- S06: the first part of the label is the module folder.

In [5]:
HEADING_RE = re.compile(r"^(#{1,6})\s+(.+?)\s*$")


def split_markdown_sections(text, title):
    """Cut Markdown text into sections at headings of level 1 to MARKDOWN_HEADING_MAX_LEVEL.

    The section label is "title > H1 > H2 ...". Headings inside code fences are ignored.
    """
    sections, stack, in_fence = [], [], False
    current = {"label": title, "start": 0, "text": "", "preamble": True}
    offset = 0
    for line in text.splitlines(keepends=True):
        if line.lstrip().startswith(("```", "~~~")):
            in_fence = not in_fence
        match = None if in_fence else HEADING_RE.match(line.rstrip("\n"))
        if match and len(match.group(1)) <= settings.MARKDOWN_HEADING_MAX_LEVEL:
            if current["text"].strip():
                sections.append(current)
            level, heading = len(match.group(1)), match.group(2)
            stack = [h for h in stack if h[0] < level] + [(level, heading)]
            path = [title] + [h for _, h in stack if h != title]
            current = {"label": " > ".join(path), "start": offset, "text": ""}
        current["text"] += line
        offset += len(line)
    if current["text"].strip():
        sections.append(current)
    return sections


def split_fenced_segments(text):
    """Cut Markdown text into [(kind, segment)]; kind is 'prose' or 'code' (a whole fenced block with its fences)."""
    segments, buffer, in_code = [], [], False
    for line in text.splitlines(keepends=True):
        is_fence = line.lstrip().startswith(("```", "~~~"))
        if is_fence and not in_code:
            if buffer:
                segments.append(("prose", "".join(buffer)))
            buffer, in_code = [line], True
        elif is_fence and in_code:
            buffer.append(line)
            segments.append(("code", "".join(buffer)))
            buffer, in_code = [], False
        else:
            buffer.append(line)
    if buffer:
        segments.append(("code" if in_code else "prose", "".join(buffer)))
    return segments


def split_code_block(block, size):
    """Keep a fenced code block whole if it fits; otherwise cut it at blank lines / lines and
    re-wrap every part in its own fences, so each chunk holds a valid code block."""
    if len(block) <= size:
        return [block]
    lines = block.strip("\n").splitlines()
    opening = lines[0]
    closing = opening.strip()[:3]
    inner = lines[1:-1] if len(lines) > 1 and lines[-1].strip().startswith(closing) else lines[1:]
    part_size = max(size - len(opening) - len(closing) - 2, 200)
    parts = split_by_lines("\n".join(inner) + "\n", part_size)
    return [f"{opening}\n{part.strip(chr(10))}\n{closing}" for part in parts]


def split_long_markdown_section(section):
    """Return [(chunk_text, None)] for one Markdown section.

    A section that fits in CHUNK_SIZE stays one chunk. A longer section without code is cut into fixed-size
    chunks. A longer section WITH code fences is cut so that a fenced code block is never torn apart:
    prose is cut into fixed-size chunks, each code block stays whole (or is split into valid smaller blocks),
    and a short paragraph is packed together with the code block next to it so an explanation stays with its code.
    """
    text = section["text"]
    if len(text.strip()) <= settings.CHUNK_SIZE:
        return [(text.strip(), None)]
    if "```" not in text and "~~~" not in text:
        return [(piece, None) for piece, _ in split_long_section(section)]

    heading, body_start = separate_heading(text)
    reserve = len(heading) + 2 if heading else 0
    atoms = []                                            # (text, index of the segment it came from)
    for number, (kind, segment) in enumerate(split_fenced_segments(text[body_start:])):
        size = settings.CHUNK_SIZE - (reserve if number == 0 else 0)
        if kind == "prose":
            for doc in make_splitter(size).create_documents([segment]):
                if doc.page_content.strip():
                    atoms.append((doc.page_content.strip(), number))
        else:
            atoms.extend((part, number) for part in split_code_block(segment.strip("\n"), size))
    if heading:
        atoms[0:1] = [((heading + "\n\n" + atoms[0][0]) if atoms else heading, atoms[0][1] if atoms else 0)]

    chunks, current, last_segment = [], "", None
    for atom, number in atoms:                            # pack neighbours from DIFFERENT segments (no overlap copies)
        if current and number != last_segment and len(current) + 2 + len(atom) <= settings.CHUNK_SIZE:
            current += "\n\n" + atom
        else:
            if current:
                chunks.append(current)
            current = atom
        last_segment = number
    if current:
        chunks.append(current)
    return [(text_, None) for text_, _ in tidy_pieces([(c, None) for c in chunks])]


def chunk_markdown_text(source_id, text, title, file, source_type):
    """Chunk one Markdown file: headings first, merge tiny sections, fixed-size fallback for long ones."""
    sections = merge_small_sections(split_markdown_sections(text, title), settings.MIN_CHUNK_CHARACTERS)
    chunks = []
    for sec in sections:
        for piece, _ in split_long_markdown_section(sec):
            chunks.append(new_chunk(source_id, sec["label"], piece, file=file, source_type=source_type))
    return chunks

In [6]:
def manifest_entries(source_id):
    """Manifest entries of one source that are files (S05 and S06 have one entry per file)."""
    return [m for m in manifest if m["source_id"] == source_id and "original_path" in m]


# --- S05: wiki pages -------------------------------------------------------
s05_chunks = []
for entry in sorted(manifest_entries("S05"), key=lambda m: m["original_path"]):
    text = (settings.CLEANED_DIR / entry["output"]).read_text(encoding="utf-8")
    page_name = Path(entry["original_path"]).stem
    s05_chunks += chunk_markdown_text("S05", text, page_name, entry["original_path"], entry["source_type"])
all_chunks["S05"] = number_chunks("S05", s05_chunks)
print(f"S05: {len(all_chunks['S05']):4d} chunks from {len(manifest_entries('S05'))} files")

S05:  308 chunks from 76 files


## 5. S06 code files (`.py`, `.sql`)
Code is **not** split by paragraphs. It is cut into structural units and the units are packed together up to `CODE_CHUNK_SIZE`:
- **Python:** a unit starts at every top-level `def` / `class` (with its decorators and the comment lines directly above it). Everything before the first definition (imports, constants) is the first unit.
- **SQL:** a unit is one statement ending in `;`. A file without any `;` (for example a dbt model) is one unit.
- A single unit longer than the limit is cut at blank lines, then at line boundaries, so a line of code is never broken in half.

Chunks keep `source_type = "code"` and the section label names the functions or the first SQL statement.

In [7]:
def split_python_units(text):
    """Cut Python source at top-level def / class (including decorators and comments right above)."""
    lines = text.splitlines(keepends=True)
    boundaries = {0}
    for i, line in enumerate(lines):
        if re.match(r"(async def |def |class )", line):
            j = i
            while j > 0 and lines[j - 1][:1] in ("@", "#"):   # decorators / comments directly above
                j -= 1
            boundaries.add(j)
    cuts = sorted(boundaries) + [len(lines)]
    units = ["".join(lines[a:b]) for a, b in zip(cuts, cuts[1:])]
    return [u for u in units if u.strip()]


def split_sql_units(text):
    """Cut SQL at statements that end with ';'. A file with no ';' is a single unit."""
    units, current = [], ""
    for line in text.splitlines(keepends=True):
        current += line
        if line.rstrip().endswith(";"):
            units.append(current)
            current = ""
    if current.strip():
        units.append(current)
    return units


def pack_units(units, max_chars):
    """Greedily pack consecutive units into chunks of at most max_chars characters."""
    chunks, current = [], ""
    for unit in units:
        if len(unit) > max_chars:
            if current.strip():
                chunks.append(current)
                current = ""
            chunks.extend(split_by_lines(unit, max_chars))
        elif len(current) + len(unit) <= max_chars:
            current += unit
        else:
            chunks.append(current)
            current = unit
    if current.strip():
        chunks.append(current)
    return [c.strip("\n") for c in chunks if c.strip()]


def code_label(chunk_text, extension, file):
    """Readable section label: function / class names for Python, "file name: first statement" for SQL."""
    if extension == ".py":
        names = re.findall(r"^(?:async def|def|class)\s+(\w+)", chunk_text, re.MULTILINE)
        if names:
            return "def/class " + ", ".join(names[:3]) + (" ..." if len(names) > 3 else "")
        return "module header (imports, constants)"
    for line in chunk_text.splitlines():
        stripped = line.strip()
        if stripped and not stripped.startswith(("--", "/*", "{#", "*")):
            return f"{Path(file).stem}: {stripped[:50]}"
    return Path(file).stem


def chunk_code_file(source_id, text, file, extension):
    """Chunk one code file by structure (never by paragraphs)."""
    units = split_python_units(text) if extension == ".py" else split_sql_units(text)
    return [new_chunk(source_id, code_label(piece, extension, file), piece, file=file, source_type="code")
            for piece in pack_units(units, settings.CODE_CHUNK_SIZE)]

In [8]:
# --- S06: course notes (Markdown) and code (.py / .sql) --------------------
s06_chunks = []
for entry in sorted(manifest_entries("S06"), key=lambda m: m["original_path"]):
    text = (settings.CLEANED_DIR / entry["output"]).read_text(encoding="utf-8")
    path = entry["original_path"]
    extension = Path(path).suffix.lower()
    module = path.split("/")[0]
    if extension == ".md":
        s06_chunks += chunk_markdown_text("S06", text, module, path, entry["source_type"])
    else:
        s06_chunks += chunk_code_file("S06", text, path, extension)
all_chunks["S06"] = number_chunks("S06", s06_chunks)

by_type = pd.Series([c["source_type"] for c in all_chunks["S06"]]).value_counts()
print(f"S06: {len(all_chunks['S06']):4d} chunks from {len(manifest_entries('S06'))} files")
print(by_type.to_string())

S06: 1000 chunks from 137 files
course_notes    946
code             54


## 6. Statistics
A good result: very few chunks below `MIN_CHUNK_CHARACTERS`, none above the size limit, and chunk lengths clustered below the limit for prose.

In [9]:
rows = []
for sid in sorted(all_chunks):
    for source_type in sorted({c["source_type"] for c in all_chunks[sid]}):
        subset = [c for c in all_chunks[sid] if c["source_type"] == source_type]
        sizes = [c["n_chars"] for c in subset]
        limit = settings.CODE_CHUNK_SIZE if source_type == "code" else settings.CHUNK_SIZE
        rows.append({
            "source": sid, "type": source_type, "chunks": len(subset),
            "mean chars": round(statistics.mean(sizes)), "median": round(statistics.median(sizes)),
            "min": min(sizes), "max": max(sizes),
            f"< {settings.MIN_CHUNK_CHARACTERS}": sum(s < settings.MIN_CHUNK_CHARACTERS for s in sizes),
            "over limit": sum(s > limit for s in sizes),
        })
stats = pd.DataFrame(rows)
print(stats.to_string(index=False))
total = sum(len(v) for v in all_chunks.values())
print(f"\nTotal chunks: {total}")

source           type  chunks  mean chars  median  min  max  < 150  over limit
   S01       textbook     319         618     658  112  791      2           0
   S02       textbook     392         584     630  104  799      4           0
   S03 lecture_slides     119         448     444  108  779      1           0
   S04       textbook     354         584     644   96  800      3           0
   S05           wiki     308         419     376   84  800      2           0
   S06           code      54         995    1116  150 1498      0           0
   S06   course_notes     946         484     495   54  800     22           0
   S07       textbook      23         563     658  118  757      1           0

Total chunks: 2515


In [10]:
# Size distribution of the prose chunks (characters), in buckets of 100
prose_sizes = [c["n_chars"] for chunks in all_chunks.values() for c in chunks if c["source_type"] != "code"]
buckets = pd.cut(pd.Series(prose_sizes), bins=range(0, settings.CHUNK_SIZE + 101, 100), right=True).value_counts().sort_index()
print("Prose chunk length (characters) -> number of chunks")
for interval, count in buckets.items():
    print(f"{str(interval):>12} {count:5d} {'#' * max(1, round(60 * count / buckets.max())) if count else ''}")

code_sizes = [c["n_chars"] for chunks in all_chunks.values() for c in chunks if c["source_type"] == "code"]
print(f"\nCode chunks: {len(code_sizes)}, median {round(statistics.median(code_sizes))} characters, max {max(code_sizes)}")

Prose chunk length (characters) -> number of chunks
    (0, 100]     9 #
  (100, 200]   173 ##################
  (200, 300]   253 ###########################
  (300, 400]   288 ##############################
  (400, 500]   300 ################################
  (500, 600]   351 #####################################
  (600, 700]   518 #######################################################
  (700, 800]   569 ############################################################
  (800, 900]     0 

Code chunks: 54, median 1116 characters, max 1498


## 7. Example chunks (for review)
Real chunks from different sources. Look for:
- Does each chunk read as a coherent piece, starting and ending in a sensible place?
- Does the `section` label describe the content?
- Where two consecutive chunks of one section are shown, is the overlap visible and useful?
- Are code chunks whole functions or statements, never cut mid-line?

In [11]:
def show_chunk(chunk, max_chars=None):
    """Print one chunk with its metadata."""
    print("=" * 100)
    print(f"{chunk['chunk_id']} | {chunk['source_type']} | difficulty={chunk['difficulty']} | {chunk['n_chars']} characters")
    print(f"section: {chunk['section']}")
    print(f"page: {chunk['page']} | file: {chunk['file']}")
    print("-" * 100)
    print(chunk["text"] if max_chars is None else chunk["text"][:max_chars])


def consecutive_in_same_section(source_id, label_start, count=2):
    """First run of `count` consecutive chunks that belong to the same (long) section."""
    chunks = all_chunks[source_id]
    for i in range(len(chunks) - count + 1):
        run = chunks[i:i + count]
        if run[0]["section"].startswith(label_start) and all(c["section"] == run[0]["section"] for c in run):
            return run
    return []

### 7.1 Textbook - S02 (OpenStax Data Science): two consecutive chunks of one long section

In [12]:
for chunk in consecutive_in_same_section("S02", "2.2"):
    show_chunk(chunk)

S02-0184 | textbook | difficulty=beginner | 747 characters
section: 2.2 Survey Design and Implementation
page: 71 | file: S02.txt
----------------------------------------------------------------------------------------------------
2.2 Survey Design and Implementation

Learning Outcomes
By the end of this section, you should be able to:
• 2.2.1 Describe the elements of survey design and identify the steps data scientists take to ensure the
reliability of survey results.
• 2.2.2 Describe methods for avoiding bias in survey questions.
• 2.2.3 Describe various sampling techniques and the advantages of each.
Surveys are a common strategy for gathering data in a wide range of domains, including market research,
social sciences, and education. Surveys collect information from a sample of individuals and often use
questionnaires to collect data. Sampling is the process of selecting a subset of a larger population to
represent and analyze information about that population.
S02-0185 | textbook |

### 7.2 Textbook - S01 (Database Design): a chunk from a chapter

In [13]:
s01_sample = [c for c in all_chunks["S01"] if c["section"].startswith("Chapter 12")]
show_chunk(s01_sample[len(s01_sample) // 2])

S01-0179 | textbook | difficulty=intermediate | 621 characters
section: Chapter 12 Normalization
page: 86 | file: S01.txt
----------------------------------------------------------------------------------------------------
Figure 12.1 Dependency diagram, by A. Watt.
The abbreviations used in Figure 12.1 are as follows:
• PD: partial dependency
• TD: transitive dependency
• FD: full dependency (Note: FD typically stands for functional dependency. Using FD as an abbreviation for full
dependency is only used in Figure 12.1.)
Boyce-Codd Normal Form (BCNF)
When a table has more than one candidate key, anomalies may result even though the relation is in 3NF. Boyce-Codd
normal form is a special case of 3NF. A relation is in BCNF if, and only if, every determinant is a candidate key.
BCNF Example 1
Consider the following table (St_Maj_Adv).


### 7.3 Markdown wiki page - S05 (all chunks of one page)

In [14]:
page = "Concepts/Data Modeling/Dimensional Modeling.md"
for chunk in [c for c in all_chunks["S05"] if c["file"] == page]:
    show_chunk(chunk)

S05-0094 | wiki | difficulty=intermediate | 543 characters
section: Dimensional Modeling
page: None | file: Concepts/Data Modeling/Dimensional Modeling.md
----------------------------------------------------------------------------------------------------
Developed by [Ralph Kimball](https://en.wikipedia.org/wiki/Ralph_Kimball), dimensional modeling is a popular technique used to model data for analytics. At it's core, dimensional modeling revolves around organizing data into two types of datasets: fact tables and dimension tables. Facts are usually comprised of numerical values that can be aggregated while dimensions hold descriptive attributes of entities/objects. A key tradeoff the dimensional model makes is it denormalizes data (increases data redundancy) in order to speed up queries.
S05-0095 | wiki | difficulty=intermediate | 302 characters
section: Dimensional Modeling
page: None | file: Concepts/Data Modeling/Dimensional Modeling.md
---------------------------------------------

### 7.4 Zoomcamp module - S06 prose (Markdown with code fences)

In [15]:
prose = [c for c in all_chunks["S06"] if c["source_type"] == "course_notes"]
with_fence = [c for c in prose if c["file"].startswith("03-data-warehouse") and "```" in c["text"]]
show_chunk(with_fence[0])
show_chunk([c for c in prose if c["file"].startswith("04-analytics-engineering")][3])

S06-0212 | course_notes | difficulty=intermediate | 791 characters
section: 03-data-warehouse > Data Warehouse and BigQuery > Public datasets
page: None | file: 03-data-warehouse/01-data-warehouse-and-bigquery.md
----------------------------------------------------------------------------------------------------
## Public datasets

Before working with BigQuery, two things worth knowing. First, BigQuery
generally caches query results — for the demos in this module caching
was disabled to get consistent results. Second, BigQuery ships a lot of
open source public data you can query out of the box.

One example is the New York City Citi Bike stations data. You can
search any dataset by table name in the search bar, and open the table
to see its columns: station id, name, short name, latitude, longitude
and others. This table has only 1,584 rows and its size is just a few
KB — a very small dataset. You can also do a quick preview to
see how the data is arranged.

Let's query it:

```sql
SEL

### 7.5 Zoomcamp code - S06 Python and SQL

In [16]:
py = [c for c in all_chunks["S06"] if c["source_type"] == "code" and c["file"].endswith(".py")
      and "07-streaming/code/src" in c["file"]]
show_chunk(max(py, key=lambda c: c["n_chars"]))          # the largest Python chunk
sql = [c for c in all_chunks["S06"] if c["source_type"] == "code" and c["file"].endswith(".sql")]
show_chunk(sql[len(sql) // 2])

S06-0989 | code | difficulty=intermediate | 1461 characters
section: def/class create_processed_events_sink_postgres, create_events_source_kafka
page: None | file: 07-streaming/code/src/job/pass_through_job.py
----------------------------------------------------------------------------------------------------
from pyflink.datastream import StreamExecutionEnvironment
from pyflink.table import EnvironmentSettings, StreamTableEnvironment


def create_processed_events_sink_postgres(t_env):
    table_name = 'processed_events'
    sink_ddl = f"""
        CREATE TABLE {table_name} (
            PULocationID INTEGER,
            DOLocationID INTEGER,
            trip_distance DOUBLE,
            total_amount DOUBLE,
            pickup_datetime TIMESTAMP
        ) WITH (
            'connector' = 'jdbc',
            'url' = 'jdbc:postgresql://postgres:5432/postgres',
            'table-name' = '{table_name}',
            'username' = 'postgres',
            'password' = 'postgres',
            

### 7.6 Lecture slides - S03 and the short article S07

In [17]:
slides = all_chunks["S03"]
show_chunk(slides[40])
show_chunk(slides[41])
show_chunk(all_chunks["S07"][1])

S03-0041 | lecture_slides | difficulty=advanced | 715 characters
section: Spark Benchmarking
page: 53 | file: S03.txt
----------------------------------------------------------------------------------------------------
Spark Benchmarking
The original papers claim Spark to be upto 100x faster
when data fits into RAM vs MapReduce, or upto 10x faster
when data is on disks
Large speedups can be observed in the RAM vs HD case
However, independent benchmarks show when both use
HDs, Spark is upto 2.5-5x faster for CPU bound workloads,
however MapReduce could still sorted faster with hard
drives:
Juwei Shi, Yunjie Qiu, Umar Farooq Minhas, Limei Jiao, Chen Wang, Berthold
Reinwald, Fatma Özcan: Clash of the Titans: MapReduce vs. Spark for Large
Scale Data Analytics. PVLDB 8(13): 2110-2121 (2015)
http://www.vldb.org/pvldb/vol8/p2110-shi.pdf
http://www.slideshare.net/ssuser6bb12d/
a-comparative-performance-evaluation-of-apache-flink
S03-0042 | lecture_slides | difficulty=advanced | 567 characters


## 8. Integrity checks

In [18]:
flat = [c for sid in sorted(all_chunks) for c in all_chunks[sid]]

ids = [c["chunk_id"] for c in flat]
empty = [c["chunk_id"] for c in flat if not c["text"].strip()]
missing_meta = [c["chunk_id"] for c in flat if not c["source_id"] or not c["licence"] or not c["section"]]
too_long_prose = [c["chunk_id"] for c in flat if c["source_type"] != "code" and c["n_chars"] > settings.CHUNK_SIZE]
too_long_code = [c["chunk_id"] for c in flat if c["source_type"] == "code" and c["n_chars"] > settings.CODE_CHUNK_SIZE]
fence_cut = [c["chunk_id"] for c in flat if c["source_type"] != "code" and c["text"].count("```") % 2 == 1]
heading_only = [c["chunk_id"] for c in flat if c["source_type"] != "code" and len(c["text"].splitlines()) == 1
                and c["text"].lstrip().startswith("#")]
tiny = [c["chunk_id"] for c in flat if c["n_chars"] < settings.DROP_CHUNKS_BELOW]

print("Total chunks                          :", len(flat))
print("Same-section merges applied           :", sum(merge_counts.values()), merge_counts)
mixed = []
for sid, chunks in all_chunks.items():
    for a, b in zip(chunks, chunks[1:]):
        if (a["source_type"] != "code" and a["source_type"] == b["source_type"] and a["file"] == b["file"]
                and a["section"] == b["section"] and len(a["text"]) + 2 + len(b["text"]) <= settings.CHUNK_SIZE):
            mixed.append((a["chunk_id"], b["chunk_id"]))
print("Neighbours of one section that could still merge:", len(mixed), "(should be 0)")
print("Duplicate chunk ids                   :", len(ids) - len(set(ids)), "(should be 0)")
print("Empty chunks                          :", len(empty), "(should be 0)")
print("Chunks missing source / licence / section:", len(missing_meta), "(should be 0)")
print(f"Prose chunks over {settings.CHUNK_SIZE} characters      :", len(too_long_prose), "(should be 0)")
print(f"Code chunks over {settings.CODE_CHUNK_SIZE} characters      :", len(too_long_code), "(should be 0)")
print("Prose chunks with an unbalanced code fence:", len(fence_cut), "(should be 0)")
print("Chunks that are only a heading line   :", len(heading_only), "(should be 0)")
print(f"Chunks shorter than {settings.DROP_CHUNKS_BELOW} characters     :", len(tiny), "(should be 0, they are dropped)")
print(f"Chunks dropped as empty (< {settings.DROP_CHUNKS_BELOW} characters):", len(dropped_chunks))
for source_id, file, text in dropped_chunks:
    print(f"   {source_id} {file}: {text!r}")

Total chunks                          : 2515
Same-section merges applied           : 98 {'S01': 18, 'S02': 30, 'S04': 26, 'S07': 2, 'S03': 6, 'S05': 3, 'S06': 13}
Neighbours of one section that could still merge: 0 (should be 0)
Duplicate chunk ids                   : 0 (should be 0)
Empty chunks                          : 0 (should be 0)
Chunks missing source / licence / section: 0 (should be 0)
Prose chunks over 800 characters      : 0 (should be 0)
Code chunks over 1500 characters      : 0 (should be 0)
Prose chunks with an unbalanced code fence: 1 (should be 0)
Chunks that are only a heading line   : 0 (should be 0)
Chunks shorter than 50 characters     : 0 (should be 0, they are dropped)
Chunks dropped as empty (< 50 characters): 2
   S05 Guides/Guides.md: 'Getting Started With Data Engineering'
   S05 Guides/SQL Guide.md: '**SELF JOIN**\n* Joins a table to itself.'


## 9. Topic and keyword labelling

### 9.1 Topic (embedding similarity)
Every chunk (its section label plus its text) and every topic description from `settings.TOPIC_DESCRIPTIONS` is turned into a vector with the embedding model from `settings.py` (`all-MiniLM-L6-v2`, runs locally, free). The chunk gets the topic whose description is most similar (cosine similarity). If even the best similarity is below `TOPIC_MIN_SIMILARITY`, the chunk is labelled `Other`: it is not really about any of the eight topics (for example Python basics in S02 or tool installation notes in S06).

Relational database design (ER modelling, normalization, keys) is described under **SQL**, because the proposal's eight topics have no separate database-design topic. This is why most of S01 is labelled SQL.

In [19]:
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(settings.EMBEDDING_MODEL)
topic_names = list(settings.TOPIC_DESCRIPTIONS)
topic_vectors = model.encode([f"{name}: {description}" for name, description in settings.TOPIC_DESCRIPTIONS.items()],
                             normalize_embeddings=True)
chunk_vectors = model.encode([f"{c['section']}\n{c['text']}" for c in flat], normalize_embeddings=True,
                             batch_size=64, show_progress_bar=False)
similarity = chunk_vectors @ topic_vectors.T           # cosine similarity, because the vectors are normalised

for chunk, row in zip(flat, similarity):
    best = int(row.argmax())
    chunk["topic_score"] = round(float(row[best]), 3)   # the stored score is also the one compared, so they always agree
    chunk["topic"] = topic_names[best] if chunk["topic_score"] >= settings.TOPIC_MIN_SIMILARITY else settings.TOPIC_OTHER_LABEL

print(f"Embedded {len(flat)} chunks with {settings.EMBEDDING_MODEL}")
print(f"Topic score: median {np.median(similarity.max(axis=1)):.2f}, "
      f"labelled '{settings.TOPIC_OTHER_LABEL}' (best score < {settings.TOPIC_MIN_SIMILARITY}): "
      f"{sum(c['topic'] == settings.TOPIC_OTHER_LABEL for c in flat)} chunks")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedded 2515 chunks with sentence-transformers/all-MiniLM-L6-v2
Topic score: median 0.41, labelled 'Other' (best score < 0.25): 266 chunks


In [20]:
# Topic distribution per source (S06 code shown separately)
pd.set_option("display.width", 200)
group = [c["source_id"] + (" code" if c["source_type"] == "code" else "") for c in flat]
distribution = pd.crosstab(pd.Series(group, name="source"), pd.Series([c["topic"] for c in flat], name="topic"),
                           margins=True, margins_name="total")
print(distribution.to_string())

topic     Big Data  Data Cleaning  Data Pipelines  Data Processing  Data Quality  Data Warehousing  ETL/ELT  Other  SQL  total
source                                                                                                                        
S01              2             26               0                0            11                12        7     38  223    319
S02             26            121               2                6            50                80       22     80    5    392
S03             44              1               3               55             1                 1        1     11    2    119
S04             46             12               7               22            31                94       24      5  113    354
S05             23              9              35               54            36                49       30     15   57    308
S06             40             31             241              295             6               230       17    

In [21]:
# The chunks labelled "Other": which source they come from and which topic they were closest to.
# These are the chunks that would have been force-fitted into a topic with a lower threshold.
other = [c for c in flat if c["topic"] == settings.TOPIC_OTHER_LABEL]
nearest = [topic_names[int(row.argmax())] for c, row in zip(flat, similarity) if c["topic"] == settings.TOPIC_OTHER_LABEL]
print(f"{len(other)} chunks labelled '{settings.TOPIC_OTHER_LABEL}' (best score < {settings.TOPIC_MIN_SIMILARITY})\n")
print(pd.crosstab(pd.Series([g for c, g in zip(flat, group) if c["topic"] == settings.TOPIC_OTHER_LABEL], name="source"),
                  pd.Series(nearest, name="nearest topic (not assigned)")).to_string())

266 chunks labelled 'Other' (best score < 0.25)

nearest topic (not assigned)  Big Data  Data Cleaning  Data Pipelines  Data Processing  Data Quality  Data Warehousing  ETL/ELT  SQL
source                                                                                                                              
S01                                  0              8               0                0             3                 3        1   23
S02                                  5             35               1                1            12                11       11    4
S03                                  4              2               1                2             1                 0        1    0
S04                                  0              3               0                1             1                 0        0    0
S05                                  1              2               1                2             2                 2        1    4
S06                 

### 9.2 Keywords (TF-IDF)
TF-IDF gives a high score to words that are frequent in one chunk but rare across all chunks, so the top terms describe what is special about that chunk. Before scoring, URLs, Markdown link targets and HTML tags are removed so that keywords are not web addresses. Single words and two-word phrases are allowed (`KEYWORD_NGRAM_RANGE`); a single word that is already part of a chosen phrase is skipped.

**Sentence-boundary guard.** The standard TF-IDF n-gram step first removes stop words and then pairs whatever words end up next to each other, so it can stitch fragments across a sentence or clause break (for example "simultaneously different"). Here a custom analyzer is used instead:
1. the text is cut at sentence and clause boundaries (`. ! ? ; : ,`, brackets, quotes, bullets, line breaks, dashes);
2. inside each piece, a two-word phrase is only formed from two words that stand **directly next to each other** in the text and are **both not stop words**.

So "the best way to store data" gives the words `way`, `store`, `data` and the phrase `store data`, but never `way store`.

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS


def text_for_keywords(text):
    """Remove URLs, Markdown link targets / images and HTML tags, which would otherwise become keywords."""
    text = re.sub(r"!\[[^\]]*\]\([^)]*\)", " ", text)          # ![image](path)
    text = re.sub(r"\]\([^)]*\)", "] ", text)                    # [link](target) -> keep the link text
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)            # bare URLs
    text = re.sub(r"<[^>]+>", " ", text)                          # HTML tags
    return text


def pick_keywords(scores, terms, n):
    """Top-n terms by score, skipping a word already inside a chosen phrase and phrases like 'str str'."""
    chosen = []
    for index in scores.argsort()[::-1]:
        if scores[index] <= 0 or len(chosen) == n:
            break
        term = terms[index]
        words = term.split()
        if len(words) == 2 and words[0] == words[1]:
            continue
        if any(term in c.split() or term == c for c in chosen):
            continue
        chosen = [c for c in chosen if c not in words]           # a phrase replaces its own single word
        chosen.append(term)
    return chosen[:n]


STOP_WORDS = set(ENGLISH_STOP_WORDS) | set(settings.KEYWORD_EXTRA_STOP_WORDS)
CLAUSE_BREAK = re.compile(r"[.!?;:,()\[\]{}<>\"“”|•\n\r\t]+|\s[-–—]\s")   # sentence and clause boundaries
WORD = re.compile(r"[a-z][a-z]+")                                       # a word of 2+ letters, no digits


def keyword_terms(text):
    """TF-IDF analyzer with a sentence-boundary guard.

    Returns the single words and two-word phrases of one chunk. The text is first cut at sentence and
    clause boundaries; a phrase is then only made from two words that are directly next to each other
    inside one piece and are both not stop words. Any other token (a number, a symbol, 'side-effect')
    also breaks the sequence, so no phrase is ever stitched across it.
    """
    terms = []
    min_n, max_n = settings.KEYWORD_NGRAM_RANGE
    for piece in CLAUSE_BREAK.split(text_for_keywords(text)):
        words = []
        for raw in piece.split():
            word = raw.strip("`*_'’\"").lower()
            words.append(word if WORD.fullmatch(word) and word not in STOP_WORDS else None)
        if min_n <= 1:
            terms.extend(w for w in words if w)
        if max_n >= 2:
            terms.extend(f"{a} {b}" for a, b in zip(words, words[1:]) if a and b)
    return terms


vectorizer = TfidfVectorizer(
    analyzer=keyword_terms,              # replaces the default tokenizer, stop-word removal and n-gram step
    min_df=settings.KEYWORD_MIN_DF,
    max_df=settings.KEYWORD_MAX_DF,
    sublinear_tf=True,
)
tfidf = vectorizer.fit_transform([c["text"] for c in flat])
terms = vectorizer.get_feature_names_out()

for i, chunk in enumerate(flat):
    chunk["keywords"] = pick_keywords(tfidf[i].toarray().ravel(), terms, settings.KEYWORDS_PER_CHUNK)

print(f"TF-IDF vocabulary: {len(terms)} terms "
      f"({sum(' ' in t for t in terms)} two-word phrases, {sum(' ' not in t for t in terms)} single words)")
print("Example: keyword_terms('The best way to store data, simultaneously. Different tools enable the concurrent jobs.')")
print("   ->", keyword_terms("The best way to store data, simultaneously. Different tools enable the concurrent jobs."))
print("Chunks with no keyword:", sum(not c["keywords"] for c in flat))

TF-IDF vocabulary: 9136 terms (3974 two-word phrases, 5162 single words)
Example: keyword_terms('The best way to store data, simultaneously. Different tools enable the concurrent jobs.')
   -> ['best', 'way', 'store', 'data', 'best way', 'store data', 'simultaneously', 'different', 'tools', 'enable', 'concurrent', 'jobs', 'different tools', 'tools enable', 'concurrent jobs']
Chunks with no keyword: 3


### 9.3 Sample of labelled chunks (for review)
15 chunks picked at random from all sources, with a fixed seed so the sample is the same on every run.

In [23]:
import random

rng = random.Random(2026)
for chunk in sorted(rng.sample(flat, 15), key=lambda c: c["chunk_id"]):
        print("=" * 100)
        print(f"{chunk['chunk_id']} | {chunk['source_type']} | topic = {chunk['topic']} (score {chunk['topic_score']})")
        print(f"section : {chunk['section']}")
        print(f"keywords: {', '.join(chunk['keywords'])}")
        print("-" * 100)
        print(chunk["text"][:300] + (" ..." if chunk["n_chars"] > 300 else ""))

S02-0102 | textbook | topic = Data Warehousing (score 0.43)
section : 1.5 Data Science with Python
keywords: python programming, python syntax, norouzi, mayfield, lawson
----------------------------------------------------------------------------------------------------
EXPLORING FURTHER
Python Programming
In the discussion below, we assume you are familiar with basic Python syntax and know how to write a
simple program using Python. If you need a refresher on the basics, please refer to Das, U., Lawson, A.,
Mayfield, C., & Norouzi, N. (2024). Introduction to Pytho ...
S02-0169 | textbook | topic = Data Warehousing (score 0.458)
section : 2.1 Overview of Data Collection Methods
keywords: observational data, transactional data, research, fields, information related
----------------------------------------------------------------------------------------------------
Types of Data
Observational and transactional data play important roles in data analysis and related decision-making, each
o

### 9.4 Spot-check per topic
For each topic: the two chunks that match it best and the weakest chunk that still got it. The weakest ones show where the topic boundary lies; `Other` shows the chunks that matched no topic.

In [24]:
for topic in topic_names + [settings.TOPIC_OTHER_LABEL]:
    members = sorted([c for c in flat if c["topic"] == topic], key=lambda c: c["topic_score"], reverse=True)
    print(f"\n=== {topic}: {len(members)} chunks")
    for label, chunk in [("best", members[0]), ("best", members[1]), ("weakest", members[-1])]:
        snippet = re.sub(r"\s+", " ", chunk["text"])[:95]
        print(f"  {label:7s} {chunk['topic_score']:.2f} {chunk['chunk_id']:9s} {chunk['section'][:45]:45s} | {snippet}")


=== SQL: 404 chunks
  best    0.72 S05-0197  Relational Database                           | A relational database organizes data into tables which can be linked together based on data tha
  best    0.70 S04-0070  8.3 Relational Database Management Systems    | 8.3 Relational Database Management Systems Learning Objectives By the end of this section, you 
  weakest 0.25 S01-0260  Chapter 16 SQL Data Manipulation Language     | SELECT ‘Total Sales’ = SUM(qty), ‘Average Sales’ = AVG(qty), stor_id FROM Sales GROUP BY StorID

=== ETL/ELT: 101 chunks
  best    0.78 S06-0322  04-analytics-engineering > Analytics Engineer | ## ETL vs ELT — a quick recap Two philosophies for getting data transformed and ready: **ETL (E
  best    0.74 S04-0216  8.6 Data Management for Shallow and Deep Lear | Data and Process Integration Traditionally, a business has various departments that have indepe
  weakest 0.26 S02-0248  2.3 Web Scraping and Social Media Data Collec | Parsing and extracting data using Py

## 10. Output

In [25]:
# Write the metadata fields in the order defined in settings, followed by the text itself
records = [{field: c[field] for field in settings.CHUNK_METADATA_FIELDS + ["text", "n_chars"]} for c in flat]
settings.CHUNKS_FILE.write_text(json.dumps(records, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Written: {settings.CHUNKS_FILE.relative_to(settings.PROJECT_ROOT)} "
      f"({settings.CHUNKS_FILE.stat().st_size / 1e6:.1f} MB, {len(flat)} chunks)")
print("Fields per chunk:", list(records[0].keys()))
print("Topics:", {topic: int(n) for topic, n in pd.Series([c["topic"] for c in flat]).value_counts().items()})

Written: data\chunks\chunks.json (2.9 MB, 2515 chunks)
Fields per chunk: ['chunk_id', 'source_id', 'source_name', 'licence', 'source_type', 'section', 'page', 'file', 'topic', 'topic_score', 'keywords', 'difficulty', 'text', 'n_chars']
Topics: {'Data Warehousing': 468, 'Data Processing': 444, 'SQL': 404, 'Data Pipelines': 290, 'Other': 266, 'Data Cleaning': 201, 'Big Data': 182, 'Data Quality': 159, 'ETL/ELT': 101}


## Documented limitations
- **S01 section labels are coarse.** The Database Design book has no numbered sub-headings, so its chunks are labelled with the chapter only (for example `Chapter 12 Normalization`), together with the page number. A pattern-based sub-heading detector was considered and rejected as too error-prone for the benefit.
- **S07 keeps line-break hyphenation** (`impor-` / `tant`). Joining hyphenated line breaks automatically could wrongly merge real compound words (for example `side-effect`), so the text is left as extracted.
- One S06 file (`01-docker-terraform/terraform/terraform_with_variable_AWS/README.md`) has a code block without a closing fence in the source itself, so one chunk shows an unbalanced fence.

## 11. Checklist before moving on
- [x] Example chunks (section 7) read well: coherent start and end, sensible `section` labels (approved).
- [x] Chunk size approved; same-section merge applied.
- [x] Statistics (section 6): few tiny chunks, none over the limit.
- [ ] Topic labels (section 9.3 / 9.4) spot-checked by hand; `TOPIC_MIN_SIMILARITY` and `TOPIC_DESCRIPTIONS` approved.
- [ ] Keywords look meaningful.
- [ ] Then continue with `03_embedding.ipynb`.